In [ ]:
from pathlib import Path
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer, AutoModelForSequenceClassification, BertTokenizer
from transformers import get_linear_schedule_with_warmup

import re

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

In [ ]:
DATA_DIR = Path("imdb-review-classification")

print("Using data folder:", DATA_DIR.resolve())

print("CSV files found:")
for file in DATA_DIR.rglob("*.csv"):
    print(file)

train_df = pd.read_csv(DATA_DIR / "train.csv")
test_df = pd.read_csv(DATA_DIR / "test.csv")

def clean_review(text):
    text = re.sub(r'<.*?>', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

train_df["review"] = train_df["review"].apply(clean_review)
test_df["review"] = test_df["review"].apply(clean_review)

# Make column names consistent: Id -> id, Review -> review, Label -> label
train_df.columns = train_df.columns.str.strip().str.lower()
test_df.columns = test_df.columns.str.strip().str.lower()

# Clean label text: Positive -> positive, Negative -> negative
train_df["label"] = train_df["label"].astype(str).str.strip().str.lower()

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

display(train_df.head())
display(test_df.head())

print("Train columns:", train_df.columns)
print("Test columns:", test_df.columns)

print("Train labels:")
print(train_df["label"].value_counts())

In [ ]:
label_to_num = {
    "negative": 0,
    "positive": 1
}

num_to_label = {
    0: "negative",
    1: "positive"
}

# Make labels clean and lowercase
train_df["label"] = train_df["label"].astype(str).str.strip().str.lower()

train_data, val_data = train_test_split(
    train_df,
    test_size=0.2,
    random_state=SEED,
    stratify=train_df["label"]
)

print("Train split:", train_data.shape)
print("Validation split:", val_data.shape)

print(train_data["label"].value_counts())
print(val_data["label"].value_counts())

In [ ]:
MODEL_NAME = "google/bert_uncased_L-2_H-256_A-4"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=False)

MAX_LEN = 512

print("Pretrained model:", MODEL_NAME)
print("Tokenizer vocab size:", len(tokenizer))
print("Pad token:", tokenizer.pad_token)
print("Pad token id:", tokenizer.pad_token_id)

In [ ]:
def encode_review(text, tokenizer, max_len=MAX_LEN):
    encoded = tokenizer(
        str(text),
        max_length=max_len,
        padding="max_length",
        truncation=True,
        return_tensors=None
    )

    return encoded["input_ids"]


class IMDBBertDataset(Dataset):
    def __init__(self, df, tokenizer, has_labels=True, max_len=MAX_LEN):
        self.df = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.has_labels = has_labels
        self.max_len = max_len

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        review = str(self.df.loc[idx, "review"])

        encoded = self.tokenizer(
            review,
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )

        item = {
            "input_ids": encoded["input_ids"].squeeze(0),
            "attention_mask": encoded["attention_mask"].squeeze(0)
        }

        if self.has_labels:
            label_text = str(self.df.loc[idx, "label"]).strip().lower()
            item["labels"] = torch.tensor(label_to_num[label_text], dtype=torch.long)

        return item

In [ ]:
def train_one_epoch(model, loader, optimizer, scheduler, device):
    model.train()

    total_loss = 0.0
    correct = 0
    total = 0

    for batch in loader:
        batch = {key: value.to(device) for key, value in batch.items()}

        optimizer.zero_grad()

        outputs = model(**batch)
        loss = outputs.loss
        logits = outputs.logits

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        total_loss += loss.item() * batch["labels"].size(0)

        preds = torch.argmax(logits, dim=1)
        labels = batch["labels"]

        correct += (preds == labels).sum().item()
        total += labels.size(0)

    avg_loss = total_loss / total
    accuracy = correct / total

    return avg_loss, accuracy


def evaluate(model, loader, device):
    model.eval()

    total_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for batch in loader:
            batch = {key: value.to(device) for key, value in batch.items()}

            outputs = model(**batch)
            loss = outputs.loss
            logits = outputs.logits

            total_loss += loss.item() * batch["labels"].size(0)

            preds = torch.argmax(logits, dim=1)
            labels = batch["labels"]

            correct += (preds == labels).sum().item()
            total += labels.size(0)

    avg_loss = total_loss / total
    accuracy = correct / total

    return avg_loss, accuracy

In [ ]:
BATCH_SIZE = 32
EPOCHS = 20
PATIENCE = 4

train_dataset = IMDBBertDataset(train_data, tokenizer, has_labels=True)
val_dataset = IMDBBertDataset(val_data, tokenizer, has_labels=True)
test_dataset = IMDBBertDataset(test_df, tokenizer, has_labels=False)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    id2label={
        0: "negative",
        1: "positive"
    },
    label2id={
        "negative": 0,
        "positive": 1
    }
).to(device)

num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print("Trainable parameters:", num_params)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=3e-5,
    weight_decay=0.01
)

total_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=total_steps // 10,  # 10% warmup
    num_training_steps=total_steps
)

train_loss_values = []
val_loss_values = []
train_acc_values = []
val_acc_values = []

best_val_acc = 0.0
best_epoch = 0
epochs_without_improvement = 0

for epoch in range(1, EPOCHS + 1):
    train_loss, train_acc = train_one_epoch(
        model,
        train_loader,
        optimizer,
        scheduler,
        device
    )

    val_loss, val_acc = evaluate(
        model,
        val_loader,
        device
    )

    train_loss_values.append(train_loss)
    val_loss_values.append(val_loss)
    train_acc_values.append(train_acc)
    val_acc_values.append(val_acc)

    print(
        f"Epoch {epoch}/{EPOCHS} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_acc:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Acc: {val_acc:.4f}"
    )

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_epoch = epoch
        epochs_without_improvement = 0
        torch.save(model.state_dict(), "best_bert_tiny_model.pt")
    else:
        epochs_without_improvement += 1

    if epochs_without_improvement >= PATIENCE:
        print(f"Early stopping at epoch {epoch}")
        break

model.load_state_dict(torch.load("best_bert_tiny_model.pt", map_location=device))
model.eval()

print("Best validation accuracy:", best_val_acc)
print("Best epoch:", best_epoch)

In [ ]:
epochs_ran = range(1, len(train_loss_values) + 1)

plt.figure(figsize=(15, 5))

plt.subplot(1, 3, 1)
plt.semilogy(epochs_ran, train_loss_values, label="Train Loss")
plt.semilogy(epochs_ran, val_loss_values, label="Validation Loss")
plt.grid(True)
plt.title("Loss Values")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()

plt.subplot(1, 3, 2)
plt.plot(epochs_ran, train_acc_values, label="Train Accuracy")
plt.hlines([0.8], 1, len(train_acc_values), colors="red", linestyles="dashed")
plt.ylim([0, 1])
plt.grid(True)
plt.title("Training Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()

plt.subplot(1, 3, 3)
plt.plot(epochs_ran, val_acc_values, label="Validation Accuracy")
plt.hlines([0.8], 1, len(val_acc_values), colors="red", linestyles="dashed")
plt.ylim([0, 1])
plt.grid(True)
plt.title("Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
UNSUP_DIR = Path("aclImdb/train/unsup")   # adjust if your folder is elsewhere

unlabeled_reviews = [f.read_text(encoding="utf-8") for f in UNSUP_DIR.glob("*.txt")]
unlabeled_df = pd.DataFrame({"review": unlabeled_reviews})
unlabeled_df["review"] = unlabeled_df["review"].apply(clean_review)

print(f"Loaded {len(unlabeled_df)} unlabeled reviews")

In [ ]:
# ── Pseudo-labeling: Step 2 – Score unlabeled reviews ─────────────────
unlabeled_dataset = IMDBBertDataset(unlabeled_df, tokenizer, has_labels=False)
unlabeled_loader  = DataLoader(unlabeled_dataset, batch_size=BATCH_SIZE, shuffle=False)

all_probs = []
model.eval()
with torch.no_grad():
    for batch in unlabeled_loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        probs = torch.softmax(model(**batch).logits, dim=1)[:, 1]
        all_probs.extend(probs.cpu().numpy())

all_probs = np.array(all_probs)
print(f"Scored {len(all_probs)} reviews")
print(f"Very confident (≥0.97 or ≤0.03): {((all_probs >= 0.97) | (all_probs <= 0.03)).sum()} reviews")

In [ ]:
# ── Pseudo-labeling: Step 3 – Filter and retrain ──────────────────────
confidence_mask = (all_probs >= 0.97) | (all_probs <= 0.03)
pseudo_df = unlabeled_df[confidence_mask].copy().reset_index(drop=True)
pseudo_df["label"] = np.where(all_probs[confidence_mask] >= 0.97, "positive", "negative")

print(f"Pseudo-labeled: {len(pseudo_df)} reviews kept")
print(pseudo_df["label"].value_counts())

# Combine with original labeled data
combined_df = pd.concat([train_df, pseudo_df], ignore_index=True)
print(f"Combined training set: {len(combined_df)} reviews")

# Re-split
train_data2, val_data2 = train_test_split(
    combined_df, test_size=0.2, random_state=SEED, stratify=combined_df["label"]
)

# New loaders
train_loader2 = DataLoader(IMDBBertDataset(train_data2, tokenizer), batch_size=BATCH_SIZE, shuffle=True)
val_loader2   = DataLoader(IMDBBertDataset(val_data2,   tokenizer), batch_size=BATCH_SIZE, shuffle=False)

# Re-initialise model from scratch (don't reuse the old weights)
model2 = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=2,
    id2label={0: "negative", 1: "positive"},
    label2id={"negative": 0, "positive": 1}
).to(device)

optimizer2 = torch.optim.AdamW(model2.parameters(), lr=3e-5, weight_decay=0.01)
total_steps2 = len(train_loader2) * EPOCHS
scheduler2 = get_linear_schedule_with_warmup(
    optimizer2,
    num_warmup_steps=total_steps2 // 10,
    num_training_steps=total_steps2
)

best_val_acc2 = 0.0
epochs_without_improvement2 = 0

for epoch in range(1, EPOCHS + 1):
    train_loss, train_acc = train_one_epoch(model2, train_loader2, optimizer2, scheduler2, device)
    val_loss,   val_acc   = evaluate(model2, val_loader2, device)

    print(f"Epoch {epoch}/{EPOCHS} | Train Acc: {train_acc:.4f} | Val Acc: {val_acc:.4f}")

    if val_acc > best_val_acc2:
        best_val_acc2 = val_acc
        epochs_without_improvement2 = 0
        torch.save(model2.state_dict(), "best_bert_pseudo.pt")
    else:
        epochs_without_improvement2 += 1

    if epochs_without_improvement2 >= PATIENCE:
        print(f"Early stopping at epoch {epoch}")
        break

model2.load_state_dict(torch.load("best_bert_pseudo.pt", map_location=device))
model2.eval()
print("Best val accuracy after pseudo-labeling:", best_val_acc2)

model = model2
val_loader = val_loader2

In [ ]:
def get_probs_and_labels(model, loader, device):
    model.eval()

    all_probs = []
    all_labels = []

    with torch.no_grad():
        for batch in loader:                         # dict, not tuple
            batch = {k: v.to(device) for k, v in batch.items()}

            outputs = model(**batch)
            probs = torch.softmax(outputs.logits, dim=1)[:, 1]  # prob of positive class

            all_probs.extend(probs.cpu().numpy())
            all_labels.extend(batch["labels"].cpu().numpy())

    return np.array(all_probs), np.array(all_labels)


val_probs, val_true = get_probs_and_labels(model, val_loader, device)

best_threshold = 0.5
best_val_acc = 0

for threshold in np.arange(0.30, 0.71, 0.01):
    val_preds = (val_probs >= threshold).astype(int)
    acc = (val_preds == val_true).mean()

    if acc > best_val_acc:
        best_val_acc = acc
        best_threshold = threshold

print(f"Best threshold: {best_threshold:.2f}")
print(f"Best validation accuracy: {best_val_acc:.4f}")

In [ ]:
def predict_review_sentiment(review_text, threshold=best_threshold):
    model.eval()

    encoded = tokenizer(
        str(review_text),
        max_length=MAX_LEN,
        padding="max_length",
        truncation=True,
        return_tensors="pt"
    )

    encoded = {key: value.to(device) for key, value in encoded.items()}

    with torch.no_grad():
        outputs = model(**encoded)
        probs = torch.softmax(outputs.logits, dim=1).squeeze(0)
        prob_positive = probs[1].item()

    label = "Positive" if prob_positive >= threshold else "Negative"

    return label, prob_positive


my_review = "I think this movie is way too short"

label, prob_positive = predict_review_sentiment(my_review)

print("Review:")
print(my_review)
print()
print("Threshold used:", best_threshold)
print("Prediction:", label)
print(f"Probability Positive: {prob_positive:.4f}")
print(f"Probability Negative: {1 - prob_positive:.4f}")

In [ ]:
def predict_labels(model, loader, threshold=0.5):
    model.eval()

    predictions = []

    with torch.no_grad():
        for batch in loader:
            batch = {key: value.to(device) for key, value in batch.items()}

            outputs = model(**batch)
            probs = torch.softmax(outputs.logits, dim=1)[:, 1]  # prob of positive class

            for prob in probs:
                label = "positive" if prob.item() >= threshold else "negative"
                predictions.append(label)

    return predictions


test_predictions = predict_labels(model, test_loader, threshold=best_threshold)

submission = pd.DataFrame({
    "Id": test_df["id"],
    "Label": test_predictions
})

submission.to_csv("prediction.csv", index=False)

display(submission.head())
print("Submission shape:", submission.shape)
print("Saved prediction.csv")
print(submission["Label"].value_counts())